In [ ]:
from src import utils

In [ ]:
from huggingface_hub import HfFolder, login

hf_token = utils.api_key_from_file("HF_KEY.txt")

HfFolder.save_token(hf_token)
login(token=hf_token)

In [ ]:
import pandas as pd

import pandas as pd
from src.data import DF_Batcher


def build_harmbench_dataset(data_path: str) -> pd.DataFrame:
    data = pd.read_json(data_path)
    data = pd.DataFrame.from_records(data["data"])
    data = data.rename(columns={"behavior": "prompt", "default_target": "target"})
    return data


train_path = "circuit-breakers-eval/data/harmbench_test_std.json"
eval_path = "circuit-breakers-eval/data/harmbench_test_std.json"

ds_train = build_harmbench_dataset(train_path)
ds_eval = build_harmbench_dataset(eval_path)

dl_train = DF_Batcher(ds_train, batch_size=10, shuffle=True)
dl_eval = DF_Batcher(ds_eval, batch_size=30, shuffle=False)

In [ ]:
print("Train dataset size:", len(ds_train))
print("Eval dataset size:", len(ds_eval))

In [ ]:
from src.eval.hb_evaluator import HarmbenchEvaluator

evaluators = [
    HarmbenchEvaluator(use_context=False, gpu_memory_utilization=0.5),
]

# evaluators = None

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from torch import optim

from src.attacks.optim_attack import OptimAttack
from src.iml_attack import IML_Attack, StopCriteria
from src.adver_model import AdverModel

# model_name = "Qwen/Qwen3-0.6B"
# model_name = "meta-llama/Llama-3.2-1B-Instruct"
model_name = "meta-llama/Llama-2-7b-chat-hf"
# model_name = "lmsys/vicuna-7b-v1.5" # TODO: not instruct model, no chat template
# model_name = "mistralai/Mistral-7B-Instruct-v0.3"
# model_name = "tiiuae/falcon-7b-instruct"
# model_name = "mosaicml/mpt-7b-chat"
# model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

utils.set_seed(42)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    # attn_implementation="sdpa",
)

torch.set_float32_matmul_precision("high")  # negligable effect

if not tokenizer.pad_token:
    tokenizer.pad_token = tokenizer.eos_token

adv_model = AdverModel(
    model=model,
    tokenizer=tokenizer,
    num_tokens=20,
)

internal_attack = OptimAttack(
    adv_model,
    optim_factory=lambda params: optim.AdamW(params, lr=1e-3),
    steps=50,
)

iml_attack = IML_Attack(
    adv_model=adv_model,
    internal_attack=internal_attack,
    optim_factory=lambda params: optim.AdamW(params, lr=2e-2),
    evaluators=evaluators,
    pred_kwargs={"max_length": 200},
)

stop = StopCriteria(
    max_epochs=5,
    max_time=15 * 60,
    patience=3,
)

In [ ]:
univ_pert = iml_attack.fit(dl_train, dl_eval, stop_criteria=stop)

In [ ]:
preds = iml_attack.predict(dl_eval, max_length=300)

for i in range(len(preds)):
    print(f" == Prompt:")
    print(ds_eval.iloc[i]["prompt"])
    print(f" == Target:")
    print(ds_eval.iloc[i]["target"])
    print(f" == Prediction:")
    print(preds[i])
    print("\n" + "=" * 50 + "\n")